# A5 — bước thử 200 mẫu (TASKS mục A5; HUONG_DAN_GPU_08-11). CHƯA train gì.
Ghép đoạn call thật sang môi trường khác (CPU, vài giây), rồi một model **chạy trên máy Kaggle** viết câu nói cho 200 mẫu và tự chấm CÓ/KHÔNG. Kết quả: tỷ lệ qua lọc và **30 dòng ngẫu nhiên** để Phương duyệt chất lượng câu **trước khi** sinh nhiều hoặc train. Theo Phương, thử **Qwen3-8B và Qwen3.5-9B** (mỗi cái một lượt chạy, 200 mẫu), rồi 14B, rồi 32B nếu cần. Khoảng 15 đến 20 phút GPU mỗi model kể cả nạp model.

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn:**
1. dataset `lifemate-code` **bản mới từ commit `404d56d`** (có hàm nạp model đa phương thức trong `gen_text.py`; bản cũ `07d4cf8` không chạy được Qwen3.5)
2. dataset `train+public_test` (raw)
3. **output của notebook A4b** (`a4b-seen-reference`): chỉ để lấy `state/processed/dev_ids.txt` (155 id dev gốc). Không cần gắn output A4
4. Model **Qwen3-8B** và/hoặc **Qwen3.5-9B** (Add Input > Models). Nên ưu tiên bản chính chủ `Qwen/...`; cell A-2 đối chiếu sha256 với Hugging Face và báo "đủ, là bản gốc" hoặc "CHƯA XÁC MINH"

**KHÔNG cần** `thuan-qwen3-4b`, `phowhisper-large-thuan`, `Hello_done`, output `train-hist`. Câu nói và call là dữ liệu cuộc thi: notebook và output để **Private**, zip kết quả chỉ gửi qua Drive riêng, không đưa vào git hay chat.
**Chạy hai model:** lượt 1 với `GEN_SIZE=8b`. Lượt 2: đổi `8b` thành `9b` ở dòng `export GEN_SIZE` trong cell A-2 rồi chạy lại **từ A-2 đến A-6** (A-3 sinh lại đúng 200 mẫu call cũ vì cùng seed; mỗi model ra một zip riêng `a5_trial_<size>_results.zip`). Chạy cả hai lượt trong **một** phiên rồi mới chạy A-7 (dọn) và Save Version. Thử 14B hoặc 32B: làm tương tự với `14b`/`32b`.
Nếu Qwen3.5-9B lỗi khi nạp: Phương dặn **báo lỗi nguyên văn** (dòng `AutoModelForCausalLM failed ...` hoặc traceback trong log cell A-4), không tự sửa mã.

In [ ]:
%%bash
# P-1: setup. Dùng nguyên kaggle_setup.sh của Phương; chỉ đặt đường dẫn giả cho Qwen3-4B và PhoWhisper
# (A5 thử không dùng hai model đó) để setup không đòi gắn hai dataset ấy.
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e 's#^QWEN=.*#QWEN=/nonexistent/qwen3-4b#' -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục (danh sách adapter từ output A4b, chỉ để biết setup chạy đúng):"; ls /kaggle/working/checkpoints | head

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra có GPU và có accelerate (gen_text.py nạp model bằng device_map="cuda")
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import transformers
print("transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn hai đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung câu nói hay call)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/a5_summary.py <<'PYEOF'
"""Tổng hợp bước thử A5: chỉ in SỐ ĐẾM (không in nội dung câu nói hay call). Ghi thêm 30 dòng ngẫu nhiên ra file để gửi Phương.
usage: python a5_summary.py <callgen_text.jsonl> <out_dir>
Chạy từ thư mục gốc repo (cần data/raw/..., src/). Ghi: <out_dir>/a5_sample30.jsonl, <out_dir>/a5_sample30.txt"""
import json
import random
import statistics
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from lifemate.callgen import app_of, text_ok  # noqa: E402
from lifemate.jsonl import read_jsonl  # noqa: E402
from lifemate.paths import split_dir  # noqa: E402
from lifemate.prompt import load_catalog  # noqa: E402

HELD = {"flights", "bike_rental", "package_tracking", "shopping_list", "podcast_radio", "loyalty"}
src, out_dir = Path(sys.argv[1]), Path(sys.argv[2])
out_dir.mkdir(parents=True, exist_ok=True)
rows = read_jsonl(src)
cat = load_catalog(split_dir("train") / "train_tools_catalog.json")
train = read_jsonl(split_dir("train") / "train.jsonl")
real_texts = {it["audio_text"].strip().lower() for it in train}


def pct(a, b):
    return f"{a}/{b} ({100 * a / b:.1f}%)" if b else f"{a}/0"


n = len(rows)
print(f"== A5 thử: {n} mẫu trong {src.name}")
ok_flags, reasons = [], Counter()
for r in rows:
    ok, why = text_ok(r["audio_text"], r, cat)
    r["_ok"], r["_why"] = ok, why
    reasons[why] += 1
    ok_flags.append(ok)
ver = [r.get("verified") for r in rows]
n_ver = sum(1 for v in ver if v)
n_ok = sum(ok_flags)
n_both = sum(1 for r in rows if r.get("verified") and r["_ok"])
print(f"model tự chấm CÓ (verified)     : {pct(n_ver, n)}")
print(f"qua luật text_ok                : {pct(n_ok, n)}")
print(f"qua CẢ HAI (sẽ được giữ khi train): {pct(n_both, n)}")
print("lý do text_ok loại:", dict(reasons.most_common()))
print("verified x text_ok :", {f"verified={a}, text_ok={b}": c for (a, b), c in
                              sorted(Counter((bool(r.get('verified')), r['_ok']) for r in rows).items())})

words = sorted(len(r["audio_text"].split()) for r in rows)
if words:
    print(f"số từ mỗi câu: min {words[0]}, trung vị {statistics.median(words)}, p90 {words[int(0.9 * (len(words) - 1))]}, max {words[-1]}")
print(f"câu rỗng: {sum(1 for r in rows if not r['audio_text'].strip())}")
print(f"câu trùng y hệt một câu THẬT trong train (chép mẫu): {pct(sum(1 for r in rows if r['audio_text'].strip().lower() in real_texts), n)}")
dup = n - len({r["audio_text"].strip().lower() for r in rows})
print(f"câu trùng nhau trong 200 mẫu: {dup}")

ncalls = Counter(len(r["label"]["tool_calls"]) for r in rows)
print("số call mỗi mẫu:", dict(sorted(ncalls.items())))
apps_per = Counter(len({app_of(c["name"]) for c in r["label"]["tool_calls"]}) for r in rows)
print("số app mỗi mẫu :", dict(sorted(apps_per.items())))
app_count = Counter(a for r in rows for a in {app_of(c["name"]) for c in r["label"]["tool_calls"]})
print("app xuất hiện nhiều nhất:", dict(app_count.most_common(8)))
held = sum(1 for r in rows if {app_of(c["name"]) for c in r["label"]["tool_calls"]} & HELD)
print(f"mẫu dùng 1 trong 6 app A4 giấu: {pct(held, n)} (khi train theo bộ chia A4 các mẫu này bị bỏ)")
if n_both:
    kept = [r for r in rows if r.get("verified") and r["_ok"]]
    print("sau lọc: số call", dict(sorted(Counter(len(r['label']['tool_calls']) for r in kept).items())),
          "| dùng app bị giấu:", sum(1 for r in kept if {app_of(c['name']) for c in r['label']['tool_calls']} & HELD))

# 30 dòng ngẫu nhiên trong TOÀN BỘ mẫu (đúng yêu cầu), kèm cờ lọc để Phương thấy luật lọc có bắt được câu xấu không
rng = random.Random(0)
pick = rng.sample(rows, min(30, n))
slim = [{"id": r["id"], "audio_text": r["audio_text"], "tool_calls": r["label"]["tool_calls"],
         "verified": r.get("verified"), "text_ok": r["_ok"], "text_ok_reason": r["_why"]} for r in pick]
with open(out_dir / "a5_sample30.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for s in slim:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
with open(out_dir / "a5_sample30.txt", "w", encoding="utf-8", newline="\n") as f:
    for i, s in enumerate(slim, 1):
        f.write(f"#{i}  verified={s['verified']}  text_ok={s['text_ok']} ({s['text_ok_reason']})\n")
        f.write(f"  CÂU : {s['audio_text']}\n")
        for c in s["tool_calls"]:
            f.write(f"  CALL: {json.dumps(c, ensure_ascii=False)}\n")
        f.write("\n")
print(f"đã ghi 30 dòng ngẫu nhiên (seed 0) -> {out_dir}/a5_sample30.txt và .jsonl (KHÔNG in nội dung ở đây)")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A-1: kiểm tra đủ đồ. Thiếu thì DỪNG. Rồi dọn BẢN SAO adapter/output A4b mà setup đã chép vào /kaggle/working (bản gốc vẫn nằm trong input).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/gen_calls.py
need scripts/gen_text.py
need src/lifemate/callgen.py
need data/processed/dev_ids.txt
need "data/raw/train+public_test/train/train.jsonl"
need "data/raw/train+public_test/train/train_tools_catalog.json"
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code (bản từ commit 783526d trở lên, có callgen), train+public_test, output notebook A4b."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
if [ "$n" != 155 ]; then echo "dev_ids.txt có $n dòng, phải là 155 (mẫu dev gốc). Gắn output A4b (có state/processed), không gắn thứ khác."; exit 1; fi
echo "dev_ids.txt: 155 id (đúng)"
rm -rf /kaggle/working/checkpoints/* /kaggle/working/outputs/*
mkdir -p outputs/logs
rm -f data/processed/callgen_*.jsonl
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-2: tìm model viết câu trong /kaggle/input theo kích thước (không đoán tên thư mục).
# GEN_SIZE: 8b = Qwen3-8B, 9b = Qwen3.5-9B (Phương muốn thử song song cả hai), rồi 14b, 32b.
# Đổi 8b thành 9b ở dòng dưới để chạy model kia (nhớ gắn model đó vào input), rồi chạy lại từ A-2 đến A-6.
export GEN_SIZE=${GEN_SIZE:-8b}
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
python -X utf8 - <<'EOF'
import hashlib
import json
import os
import sys
from pathlib import Path

# (model_type, hidden_size, số tầng văn bản). Qwen3.5 là model đa phương thức: cấu hình văn bản nằm trong "text_config".
WANT = {"8b": ("qwen3", 4096, 36), "9b": ("qwen3_5", 4096, 32), "14b": ("qwen3", 5120, 40), "32b": ("qwen3", 5120, 64)}
# sha256 các shard của bản gốc trên Hugging Face (Qwen/Qwen3-8B và Qwen/Qwen3.5-9B), tra theo kích thước file vì Kaggle có thể đổi tên shard.
OFFICIAL = {
    "8b": {3996250744: "31d6a825ae35f11fb85b195b4c42c146c051e446433125a215336abdf95cbf5f",
           3993160032: "5991236cea6fe21f3d43cab0f0e84448734fbbe0789816202989f2ddc9d18282",
           3959604768: "c5185c4794be2d8a9784d5753c9922db38df478ce11f9ed0b415b7304d896836",
           3187841392: "b5ee7de71fbf17db3d5704e0c8f2bc7d005ca9e1d7ca2aeb19827b0cfcaa917a",
           1244659840: "20c2d6366ab85c90786ccdd829cd2b9e7d30ef3b2ebbb998280e7e4014b542ff"},
    "9b": {5276436216: "db6f444b43d318c92f360a13a25561a6a65b10c0631b8ed305a426dbaa6c380e",
           5335161512: "31c7d7e2dd5d207840b31cc59083c8f4c4718959149e0358c0364052bb9a0330",
           5368717440: "7ec36ba3a4176a44c3c0876ad80c56a2f70c84bf008d82e9501df642f17dadec",
           3325995712: "b62b0c4cd7e44edee103ee8f4fe225f246d5e768e07bfd5f25b63a8aa1fdd0c6"},
}
size_key = os.environ["GEN_SIZE"]
want = WANT[size_key]
found = []
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    if not str(c.get("model_type", "")).startswith("qwen3"):
        continue
    t = c.get("text_config", c)
    weights = sorted(p.parent.glob("*.safetensors"))
    gb = sum(w.stat().st_size for w in weights) / 1e9
    found.append((p.parent, c.get("model_type"), t.get("hidden_size"), t.get("num_hidden_layers"), len(weights), gb))
print("Các model qwen3* đang gắn vào notebook:")
for d, mt, h, L, nw, gb in found:
    print(f"  {d}  type={mt} hidden={h} layers={L}  {nw} file safetensors, {gb:.1f} GB")
pick = [f for f in found if (f[1], f[2], f[3]) == want and f[4] > 0]
if not pick:
    sys.exit(f"KHÔNG thấy model GEN_SIZE={size_key} (type={want[0]}, hidden={want[1]}, layers={want[2]}) có file safetensors. "
             "Add Input > Models > tìm đúng model rồi chạy lại cell này. Nếu model có trong danh sách trên nhưng khác số, "
             "báo lại nguyên văn danh sách đó.")
d = pick[0][0]
cfg = (d / "config.json").read_bytes()
with open("/kaggle/working/a5_env.sh", "w", encoding="utf-8") as f:
    f.write(f'export GEN_SIZE={size_key}\nexport GEN="{d}"\n')
with open("/kaggle/working/gen_model_info.txt", "w", encoding="utf-8") as f:
    f.write(f"GEN_SIZE={size_key}\nthư mục: {d}\nconfig.json sha256: {hashlib.sha256(cfg).hexdigest()}\n")
    f.write(f"model_type={pick[0][1]} hidden_size={pick[0][2]} num_hidden_layers={pick[0][3]} tổng safetensors={pick[0][5]:.2f} GB\n")
    for w in sorted(d.glob("*.safetensors")):
        f.write(f"  {w.name}  {w.stat().st_size} bytes\n")
    f.write("tệp trong thư mục: " + ", ".join(sorted(x.name for x in d.iterdir())) + "\n")
    # Đối chiếu sha256 với bản gốc. Không khớp KHÔNG có nghĩa là bị sửa (có thể chia lại shard), nhưng khi đó chưa xác minh là bản gốc.
    ref = OFFICIAL.get(size_key)
    if ref:
        verdict, matched = [], set()
        for w in sorted(d.glob("*.safetensors")):
            sz = w.stat().st_size
            if sz not in ref:
                verdict.append(f"{w.name}: kích thước KHÁC bản gốc (không đối chiếu được)")
                continue
            h = hashlib.sha256()
            with open(w, "rb") as fh:
                while block := fh.read(1 << 24):
                    h.update(block)
            ok = h.hexdigest() == ref[sz]
            if ok:
                matched.add(sz)
            verdict.append(f"{w.name}: " + ("TRÙNG bản gốc" if ok else "KHÁC bản gốc (sha256 không trùng)"))
        full = matched == set(ref) and len(list(d.glob("*.safetensors"))) == len(ref)
        head = (f"ĐỐI CHIẾU với bản gốc Hugging Face: {len(matched)}/{len(ref)} shard trùng sha256 (đủ, là bản gốc)" if full else
                f"ĐỐI CHIẾU với bản gốc Hugging Face: CHƯA XÁC MINH ({len(matched)}/{len(ref)} shard trùng)")
        f.write(head + "\n" + "\n".join("  " + v for v in verdict) + "\n")
        print(head)
print("\nDÙNG:", d)
EOF
cat /kaggle/working/gen_model_info.txt

In [ ]:
%%bash
# A-3 (CPU, vài giây): sinh 200 mẫu call bằng ghép đoạn call thật. Seed 42 cố định nên mọi kích thước model nhận đúng 200 mẫu giống nhau.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/gen_calls.py --n 200 --seed 42 2>&1 | tee outputs/logs/a5_gen_calls.txt
wc -l data/processed/callgen_pending.jsonl

In [ ]:
%%bash
# A-4 (GPU): model viết câu cho 200 mẫu, rồi tự trả lời CÓ/KHÔNG. Chạy trên máy Kaggle, không gọi API ngoài.
# Log chỉ hiện sau khi cell xong; nạp model 8B mất vài phút. Chỉ in số đếm tiến độ.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
echo "model: $GEN ($GEN_SIZE)"
OUT=data/processed/callgen_text_${GEN_SIZE}.jsonl
rm -f "$OUT"
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
START=$(date +%s)
python -X utf8 scripts/gen_text.py --model "$GEN" --inp data/processed/callgen_pending.jsonl --out "$OUT" --batch 32 2>&1 | tee outputs/logs/a5_gen_text_${GEN_SIZE}.txt
echo "thời gian gen_text: $(( $(date +%s) - START )) s (gồm nạp model)" | tee -a outputs/logs/a5_gen_text_${GEN_SIZE}.txt
wc -l "$OUT"
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
%%bash
# A-5: tổng hợp bằng số đếm (tỷ lệ qua lọc, độ dài, cơ cấu call) và ghi 30 dòng ngẫu nhiên ra file để gửi Phương. KHÔNG in nội dung câu.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
python -X utf8 /kaggle/working/helpers/a5_summary.py data/processed/callgen_text_${GEN_SIZE}.jsonl /kaggle/working/a5_out_${GEN_SIZE} 2>&1 | tee outputs/logs/a5_summary_${GEN_SIZE}.txt
ls -la /kaggle/working/a5_out_${GEN_SIZE}

In [ ]:
%%bash
# A-6: gói kết quả nhỏ để gửi Phương: a5_trial_<size>_results.zip (200 mẫu có câu, 30 dòng để duyệt, log, môi trường, thông tin model)
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a5_trial_${GEN_SIZE}_results.zip \
  data/processed/callgen_text_${GEN_SIZE}.jsonl \
  /kaggle/working/a5_out_${GEN_SIZE}/a5_sample30.txt /kaggle/working/a5_out_${GEN_SIZE}/a5_sample30.jsonl \
  outputs/logs/a5_*.txt /kaggle/working/torch_cuda.txt /kaggle/working/gen_model_info.txt
ls -la /kaggle/working/*.zip

In [ ]:
%%bash
# A-7 (CUỐI CÙNG): dọn thư mục làm việc (xóa lifemate/ có symlink tới dữ liệu đọc-chỉ), giữ zip, câu đã sinh và log. Rồi Save Version.
cd /kaggle/working
mkdir -p a5_keep
cp lifemate/data/processed/callgen_*.jsonl a5_keep/ 2>/dev/null || true
rm -rf lifemate helpers setup_patched.sh config_paths.txt checkpoints
du -sh /kaggle/working/* 2>/dev/null
echo "xong. Bây giờ Save Version (private). Lấy a5_trial_*_results.zip ở tab Output."